# EWC 2026 Dota 2 fact-agent, clean version

Этот ноутбук — компактная рабочая версия поверх одной SQLite-базы. Он использует уже собранные и очищенные данные проекта:

- официальные никнеймы и позиции игроков;
- fantasy-очки по каждой карте;
- role-slot агрегаты `core_pair`, `mid_single`, `support_pair`;
- reliability-v2 score 1-100;
- stage metadata: `stage_name`, `stage_bucket`, `is_group_stage_bucket`, `is_main_playoff`;
- coverage/status представления для backfill и replay-derived данных;
- source-first агент, который сначала ищет факт в SQLite, а не выдумывает ответ.

Основной файл базы после локальной сборки: `data/ewc_2026_fantasy_compact.sqlite` или `data/db/ewc_2026_fantasy_compact.sqlite`.


In [ ]:
from pathlib import Path
import importlib
import os
import sys
import sqlite3
import pandas as pd

# Layout switch:
# - "flat_colab"  : files uploaded into one flat /content directory
# - "project"     : normal repository structure with src/ and data/
NOTEBOOK_LAYOUT = "flat_colab"

# Optional manual overrides.
CUSTOM_PROJECT_ROOT = None
CUSTOM_SRC_DIR = None
CUSTOM_DB_PATH = None

# Ranking filter switch for notebook analytics.
# True  -> rankings only show teams qualified to TI 2026
# False -> include all teams present in the EWC database
TI_QUALIFIED_ONLY = True

DEFAULT_DB_FILENAME = "ewc_2026_fantasy_compact.sqlite"


def candidate_db_paths(root: Path) -> list[Path]:
    return [
        root / DEFAULT_DB_FILENAME,
        root / 'data' / DEFAULT_DB_FILENAME,
        root / 'data' / 'db' / DEFAULT_DB_FILENAME,
    ]


def resolve_existing_db(candidates: list[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return None


def resolve_project_root() -> Path:
    if CUSTOM_PROJECT_ROOT:
        root = Path(CUSTOM_PROJECT_ROOT).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"CUSTOM_PROJECT_ROOT does not exist: {root}")
        return root

    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        Path('/content'),
        Path('/content/fantasy-analytics'),
        Path('/content/project'),
        Path('/content/drive/MyDrive/fantasy-analytics'),
    ]

    for candidate in candidates:
        src_ok = (candidate / 'src' / 'ewc_fact_agent_tools.py').exists()
        db_ok = resolve_existing_db(candidate_db_paths(candidate)) is not None
        if src_ok and db_ok:
            return candidate

    checked = '\n - '.join(str(p) for p in candidates)
    raise FileNotFoundError(
        'Could not resolve project layout automatically. ' +
        'Set CUSTOM_PROJECT_ROOT manually. Checked:\n - ' + checked
    )


if NOTEBOOK_LAYOUT == 'flat_colab':
    PROJECT_ROOT = Path(CUSTOM_PROJECT_ROOT or '/content').expanduser().resolve()
    SRC_DIR = Path(CUSTOM_SRC_DIR or '/content').expanduser().resolve()
    if CUSTOM_DB_PATH:
        DB_PATH = Path(CUSTOM_DB_PATH).expanduser().resolve()
    else:
        DB_PATH = resolve_existing_db(candidate_db_paths(PROJECT_ROOT))
elif NOTEBOOK_LAYOUT == 'project':
    PROJECT_ROOT = resolve_project_root()
    SRC_DIR = Path(CUSTOM_SRC_DIR).expanduser().resolve() if CUSTOM_SRC_DIR else (PROJECT_ROOT / 'src').resolve()
    if CUSTOM_DB_PATH:
        DB_PATH = Path(CUSTOM_DB_PATH).expanduser().resolve()
    else:
        DB_PATH = resolve_existing_db(candidate_db_paths(PROJECT_ROOT))
else:
    raise ValueError("NOTEBOOK_LAYOUT must be 'flat_colab' or 'project'")

if DB_PATH is None or not DB_PATH.exists():
    raise FileNotFoundError(
        'Database file not found. Checked common locations such as ' +
        '/content/ewc_2026_fantasy_compact.sqlite, /content/data/..., /content/data/db/... '
        'or project data/ and data/db/. Set CUSTOM_DB_PATH manually if needed.'
    )
if not (SRC_DIR / 'ewc_fact_agent_tools.py').exists():
    raise FileNotFoundError(f"ewc_fact_agent_tools.py not found in: {SRC_DIR}")

try:
    os.chmod(DB_PATH, 0o666)
except OSError:
    pass

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

importlib.invalidate_caches()
import ewc_fact_agent_tools


def patched_connect(db_path=None):
    con = sqlite3.connect(str(DB_PATH))
    con.row_factory = sqlite3.Row
    return con


ewc_fact_agent_tools.connect = patched_connect
ewc_fact_agent_tools.DB_PATH = DB_PATH

from ewc_fact_agent_tools import (
    EWCFactAgent,
    db_status,
    ask,
    roster,
    top_fantasy_maps,
    player_maps,
    role_map_summary,
    reliable_players_v2,
    reliable_role_slots_v2,
    reliability_backtest_v2,
    ti_qualified_teams,
    source_cache_status,
    banner_optimizer_players,
    banner_optimizer_role_slots,
    scoring_formula,
    source_urls,
    explain_sql_plan,
    explain_system_short,
)

agent = EWCFactAgent(DB_PATH)


def ask_v2(question: str, max_rows: int | None = None, use_llm: bool = False):
    """Main helper: returns AgentResult and prints markdown answer."""
    result = agent.ask(question, max_rows=max_rows, use_llm=use_llm)
    print(result.answer_markdown)
    return result


def sql_df(sql: str, params=None) -> pd.DataFrame:
    con = sqlite3.connect(DB_PATH)
    try:
        return pd.read_sql_query(sql, con, params=params or [])
    finally:
        con.close()


print('Clean EWC 2026 fact-agent ready.')
print(f'Layout mode: {NOTEBOOK_LAYOUT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'SRC_DIR: {SRC_DIR}')
print(f'DB_PATH: {DB_PATH}')
print(f'TI_QUALIFIED_ONLY: {TI_QUALIFIED_ONLY}')


        ## 1. Быстрая проверка базы

        Если все ключевые объекты существуют, можно пользоваться агентом.
        


In [ ]:
# Глубокая проверка подключения
import sqlite3
import os

try:
    # 1. Пробуем открыть файл средствами Python напрямую
    with open(DB_PATH, 'rb') as f:
        header = f.read(100)
        print(f"Файл читается как системный объект. Заголовок: {header[:15]}...")

    # 2. Пробуем открыть через sqlite3 с явным URI
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=rw", uri=True)
    conn.execute("SELECT 1").fetchone()
    conn.close()
    print("Прямое подключение через sqlite3: Успешно!")

    # 3. Проверка через обертку агента
    status = db_status()
    display(status)
except Exception as e:
    print(f"Критическая ошибка: {e}")
    print(f"DB_PATH: {DB_PATH}")
    print(f"Доступные файлы: {os.listdir('/content')}")

        ## 2. Основной агент

        Агент сначала пытается решить вопрос через SQLite. Если в вопросе есть внешний фильтр вроде `TI 2026 qualification`, он не делает вид, что этот список есть в базе, а просит сверить источник.
        


In [ ]:
examples = [
    'какой был состав у BetBoom?',
    'укажи топ 15 лучших фэнтези игроков 1 позиции, их команды и лучшие результаты',
    'укажи топ 15 лучших фэнтези игроков 1 позиции из команд, отобравшихся на TI 2026',
    'покажи надежные core_pair слоты для TI 2026',
    'покажи надежных мидеров для фэнтези',
    'какие stage_bucket есть в базе и чем group stage отличается от playoffs?',
    'покажи покрытие backfill-метрик и какие из них source-limited',
    'есть ли replay-derived данные по турниру и насколько они полные?',
    'подробно объясни как считались fantasy очки',
]

for q in examples[:4]:
    print('\nQUESTION:', q)
    _ = ask_v2(q, max_rows=8)


        ## 3. Прямые SQL-friendly helpers

        Эти функции удобны, когда не нужен natural language router.
        


In [ ]:
# Принудительное использование глобального агента и патча путей
display(roster("Team Falcons"))
display(top_fantasy_maps(position=1, limit=10))
display(top_fantasy_maps(position=1, ti2026_only=True, limit=10))
display(reliable_players_v2(position=1, limit=10))
display(reliable_role_slots_v2(role_slot="core_pair", limit=10))
display(ti_qualified_teams())
display(reliability_backtest_v2())

        ## 4. Fantasy banner optimizer

        Optimizer использует текущий fantasy-профиль и оценивает привлекательность пика по повторяемому потолку, а не по простой средней карте.

        По умолчанию саппорты исключены из рекомендаций, потому что их статистика в этой базе low-confidence.
        


In [ ]:
# Оптимизация теперь должна работать с корректным DB_PATH
display(banner_optimizer_players(position=1, ti2026_only=True, limit=15))
display(banner_optimizer_role_slots(role_slot="core_pair", ti2026_only=True, limit=10))

# Natural language route:
_ = ask_v2("оптимизируй баннер для игроков 1 позиции из команд отобравшихся на TI 2026", max_rows=10)

        ## 5. Конструктор fantasy-баннеров

        Можно создать профиль под любые выпавшие коэффициенты. Профиль сохранится в тех же таблицах:

        - `fantasy_scoring_profiles`;
        - `fantasy_scoring_profile_stats`;
        - `fantasy_scoring_profile_banners`;
        - `fantasy_player_map_scores`;
        - `fantasy_team_role_map_scores`;
        - `fantasy_pick_value`.

        Ячейка ниже безопасная: пример не запускается автоматически.
        


In [ ]:
from fantasy_profile_constructor import create_or_replace_banner_profile

MY_CUSTOM_BANNER = {
    "core": [
        ("kills", 2.5),
        ("creep_score", 2.5),
        ("teamfight_participation", 1.8),
    ],
    "mid": [
        ("creep_score", 2.7),
        ("runes_grabbed", 1.8),
        ("teamfight_participation", 2.7),
    ],
    "support": [
        ("lotus", 3.2),
        ("watchers_taken", 2.1),
        ("teamfight_participation", 1.5),
    ],
}

# Чтобы реально создать профиль, поменяй False на True.
if False:
    con = sqlite3.connect(DB_PATH)
    profile_id = create_or_replace_banner_profile(
        con,
        "my_new_banner_profile",
        MY_CUSTOM_BANNER,
        profile_name="My new fantasy banner",
        set_default=False,  # True сделает профиль дефолтным
    )
    con.close()
    print("created:", profile_id)


        ## 6. Web/source tools

        Source-cache уже хранит OpenDota heroes и TI 2026 qualified teams. Для новых внешних фактов агент все равно не должен фантазировать: сначала источник, потом SQL-фильтр.
        


In [ ]:
display(source_cache_status())
display(ti_qualified_teams())
display(source_urls('команды отобравшиеся на TI 2026'))

coverage_df = sql_df("""
SELECT
    stat_name,
    preferred_source,
    coverage_status,
    has_stage_evidence,
    is_row_complete,
    nonzero_raw_rows,
    source_missing_rows,
    objective_derived_rows
FROM analytics_fantasy_backfill_coverage
ORDER BY
    CASE coverage_status
        WHEN 'filled_backfill' THEN 1
        WHEN 'filled_approximation' THEN 2
        WHEN 'source_needed' THEN 3
        ELSE 4
    END,
    stat_name
""")
display(coverage_df)

replay_summary_df = sql_df("""
SELECT
    source_name,
    stat_name,
    distinct_matches,
    row_count,
    nonzero_rows,
    max_raw_value
FROM analytics_replay_metric_summary
ORDER BY stat_name
""")
display(replay_summary_df)

match_coverage_df = sql_df("""
SELECT
    source_name,
    COUNT(*) AS covered_matches,
    ROUND(AVG(nonzero_rows), 2) AS avg_nonzero_rows_per_match,
    MAX(last_tick) AS max_last_tick
FROM analytics_replay_match_coverage
GROUP BY source_name
ORDER BY covered_matches DESC
""")
display(match_coverage_df)


        ## 7. SQL planner and confidence intervals

        `explain_sql_plan(...)` shows which deterministic route, views, filters and SQL template the agent will use. This is the fastest way to debug complex questions before letting GigaChat polish the answer.

        Reliability-v2 rows now include interval columns: `low_estimate`, `expected_estimate`, `high_estimate`, `uncertainty_score`, `confidence_label`.
        


In [ ]:
# SQL planner, reliability, role slots, and stage-aware sample views
display(explain_sql_plan('top 15 fantasy pos1 players from TI 2026 qualified teams'))

player_cols = [
    'reliability_score_1_100',
    'official_name',
    'team_name',
    'official_position',
    'predicted_score_raw',
    'low_estimate',
    'expected_estimate',
    'high_estimate',
    'uncertainty_score',
    'confidence_label',
]
role_cols = [
    'reliability_score_1_100',
    'team_name',
    'role_slot',
    'player_names',
    'predicted_score_raw',
    'low_estimate',
    'expected_estimate',
    'high_estimate',
    'confidence_label',
]

try:
    display(reliable_players_v2(position=1, ti2026_only=True, limit=10)[player_cols])
    display(reliable_role_slots_v2(role_slot='core_pair', ti2026_only=True, limit=10)[role_cols])
    display(reliable_role_slots_v2(role_slot='mid_single', ti2026_only=True, limit=10)[role_cols])
except Exception as e:
    print(f'Ошибка вывода reliability-блоков: {e}')

stage_demo_df = sql_df("""
SELECT
    match_id,
    match_date,
    team_name,
    official_name,
    official_position,
    stage_name,
    stage_bucket,
    is_group_stage_bucket,
    is_main_playoff,
    fantasy_score
FROM analytics_player_maps
WHERE ti2026_qualified = 1
ORDER BY match_date DESC, team_name, official_position
LIMIT 12
""")
display(stage_demo_df)


        ## 8. Dashboard and regression tests

        Dashboard is intentionally kept as a separate file so the notebook stays compact.

        - Dashboard file: `../dashboard/app.py`
        - Tests file: `../tests/regression_tests.py`
        


In [ ]:
# В Colab используем плоскую структуру /content/ для всех файлов
DASHBOARD_PATH = "/content/app.py"
TESTS_PATH = "/content/regression_tests.py"

print("Dashboard launch command:")
print(f"streamlit run {DASHBOARD_PATH}")

print("\nRegression tests launch command:")
print(f"{sys.executable} {TESTS_PATH}")

# Оригинальные относительные пути закомментированы:
# !python "../tests/regression_tests.py"

        ## 9. Optional GigaChat post-processing

        По умолчанию ответы deterministic. Если в Colab/окружении есть `GIGACHAT_CREDENTIALS`, можно вызвать:

        ```python
        ask_v2("покажи надежных игроков для фэнтези", use_llm=True)
        ```

        LLM получает только черновик и таблицы, поэтому не должна добавлять новые числа вне данных.
        


In [ ]:
# Интерактивный режим:
# chat(use_llm=False)

# В конце работы можно закрыть соединение:
# agent.close()


# Раздел тестирования

In [ ]:
# ## 3.1 Рейтинг fantasy-показателей для core_pair при коэффициенте 1.0

import sqlite3
import pandas as pd

con = sqlite3.connect(DB_PATH)

sql = """
WITH core_pair_maps AS (
    SELECT
        f.match_id,
        f.team_name
    FROM player_game_fantasy_summary f
    JOIN player_identity_registry pir
      ON pir.account_id = f.account_id
     AND pir.team_name = f.team_name
    WHERE pir.official_position IN (1, 3)
    GROUP BY f.match_id, f.team_name
    HAVING COUNT(DISTINCT pir.official_position) = 2
),
core_pair_stat_points AS (
    SELECT
        c.match_id,
        c.team_name,
        sc.stat_name,
        COALESCE(sc.emblem_color, 'unknown') AS color_group,
        AVG(COALESCE(sp.base_points, 0.0)) AS core_pair_stat_points_x1
    FROM core_pair_maps c
    JOIN player_game_fantasy_summary f
      ON f.match_id = c.match_id
     AND f.team_name = c.team_name
    JOIN player_identity_registry pir
      ON pir.account_id = f.account_id
     AND pir.team_name = f.team_name
     AND pir.official_position IN (1, 3)
    JOIN fantasy_scoring_stat_catalog sc
      ON 1 = 1
    LEFT JOIN fantasy_player_map_stat_points sp
      ON sp.match_id = f.match_id
     AND sp.account_id = f.account_id
     AND sp.team_name = f.team_name
     AND sp.stat_name = sc.stat_name
    GROUP BY
        c.match_id,
        c.team_name,
        sc.stat_name,
        sc.emblem_color
),
ranked AS (
    SELECT
        stat_name,
        color_group,
        core_pair_stat_points_x1,
        ROW_NUMBER() OVER (
            PARTITION BY stat_name
            ORDER BY core_pair_stat_points_x1
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY stat_name
        ) AS cnt
    FROM core_pair_stat_points
)
SELECT
    stat_name,
    color_group,
    COUNT(*) AS core_pair_maps,
    ROUND(AVG(core_pair_stat_points_x1), 2) AS avg_fantasy_points_x1,
    ROUND(MAX(core_pair_stat_points_x1), 2) AS max_fantasy_points_x1,
    ROUND(
        AVG(
            CASE
                WHEN rn IN (
                    CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                    CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                )
                THEN core_pair_stat_points_x1
            END
        ),
        2
    ) AS p75_fantasy_points_x1
FROM ranked
GROUP BY stat_name, color_group
ORDER BY avg_fantasy_points_x1 DESC, stat_name
"""

core_pair_stat_ranking = pd.read_sql_query(sql, con)

display(core_pair_stat_ranking)

for color in ["red", "blue", "green"]:
    print(f"\n{color.upper()}")
    display(
        core_pair_stat_ranking
        .query("color_group == @color")
        .reset_index(drop=True)
    )

con.close()


In [ ]:
# ## 3.1 Рейтинг fantasy-показателей для всех ролей при коэффициенте 1.0

import sqlite3
import pandas as pd

def get_stat_ranking(positions, group_label):
    con = sqlite3.connect(DB_PATH)

    # Определяем условия фильтрации: для пар нужно наличие обоих игроков в матче, для мидера - один игрок
    pos_list = ", ".join(map(str, positions))
    count_check = f"HAVING COUNT(DISTINCT pir.official_position) = {len(positions)}"

    sql = f"""
    WITH target_maps AS (
        SELECT f.match_id, f.team_name
        FROM player_game_fantasy_summary f
        JOIN player_identity_registry pir ON pir.account_id = f.account_id AND pir.team_name = f.team_name
        WHERE pir.official_position IN ({pos_list})
        GROUP BY f.match_id, f.team_name
        {count_check}
    ),
    stat_points AS (
        SELECT
            c.match_id, c.team_name, sc.stat_name,
            COALESCE(sc.emblem_color, 'unknown') AS color_group,
            AVG(COALESCE(sp.base_points, 0.0)) AS avg_points_x1
        FROM target_maps c
        JOIN player_game_fantasy_summary f ON f.match_id = c.match_id AND f.team_name = c.team_name
        JOIN player_identity_registry pir ON pir.account_id = f.account_id AND pir.official_position IN ({pos_list})
        JOIN fantasy_scoring_stat_catalog sc ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp ON sp.match_id = f.match_id AND sp.account_id = f.account_id AND sp.stat_name = sc.stat_name
        GROUP BY c.match_id, c.team_name, sc.stat_name, sc.emblem_color
    ),
    ranked AS (
        SELECT
            stat_name, color_group, avg_points_x1,
            ROW_NUMBER() OVER (PARTITION BY stat_name ORDER BY avg_points_x1) AS rn,
            COUNT(*) OVER (PARTITION BY stat_name) AS cnt
        FROM stat_points
    )
    SELECT
        stat_name, color_group,
        ROUND(AVG(avg_points_x1), 2) AS avg_fantasy_points_x1,
        ROUND(MAX(avg_points_x1), 2) AS max_fantasy_points_x1,
        ROUND(AVG(CASE WHEN rn IN (CAST(((cnt - 1) * 0.75) AS INTEGER) + 1, CAST(((cnt - 1) * 0.75) AS INTEGER) + 2) THEN avg_points_x1 END), 2) AS p75_fantasy_points_x1
    FROM ranked
    GROUP BY stat_name, color_group
    ORDER BY p75_fantasy_points_x1 DESC
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df

for label, pos in [("CORE PAIR (1+3)", [1, 3]), ("MID (2)", [2]), ("SUPPORT PAIR (4+5)", [4, 5])]:
    print(f"\n{'='*20} {label} {'='*20}")
    ranking = get_stat_ranking(pos, label)
    display(ranking.head(10))

In [ ]:
# ## 3.2 Ранжирование командных сочетаний по каждой статистике для role-слотов core_pair / mid_single / support_pair

import sqlite3
import pandas as pd

ROLE_RANKING_CONFIG = {
    "core_pair": {
        "label": "CORE PAIR COMBINATIONS (positions 1 and 3)",
        "positions": [1, 3],
    },
    "mid_single": {
        "label": "MID OPTIONS (position 2)",
        "positions": [2],
    },
    "support_pair": {
        "label": "SUPPORT PAIR COMBINATIONS (positions 4 and 5)",
        "positions": [4, 5],
    },
}


def _build_role_slot_context(positions, ti_qualified_only=TI_QUALIFIED_ONLY):
    pos_list = ", ".join(map(str, positions))
    expected_positions = len(positions)
    team_filter_sql = """
        AND EXISTS (
            SELECT 1
            FROM analytics_ti2026_teams ti
            WHERE ti.team_name = pir.team_name
        )
    """ if ti_qualified_only else ""
    return pos_list, expected_positions, team_filter_sql


def get_role_slot_stat_ranking(positions, role_key, min_maps=1, ti_qualified_only=TI_QUALIFIED_ONLY):
    con = sqlite3.connect(DB_PATH)
    pos_list, expected_positions, team_filter_sql = _build_role_slot_context(
        positions,
        ti_qualified_only=ti_qualified_only,
    )
    sql = f"""
    WITH role_players AS (
        SELECT
            pir.account_id,
            pir.team_name,
            pir.official_name,
            pir.official_position,
            pir.role_group
        FROM player_identity_registry pir
        WHERE pir.official_position IN ({pos_list})
        {team_filter_sql}
    ),
    role_names AS (
        SELECT
            rp.team_name,
            GROUP_CONCAT(rp.official_name, ', ') AS player_names,
            MIN(rp.role_group) AS role_group
        FROM (
            SELECT team_name, official_name, official_position, role_group
            FROM role_players
            ORDER BY team_name, official_position
        ) rp
        GROUP BY rp.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    target_maps AS (
        SELECT
            f.match_id,
            f.team_name
        FROM player_game_fantasy_summary f
        JOIN role_players rp
          ON rp.account_id = f.account_id
         AND rp.team_name = f.team_name
        GROUP BY f.match_id, f.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    role_stat_maps AS (
        SELECT
            tm.match_id,
            tm.team_name,
            '{role_key}' AS role_slot,
            sc.stat_name,
            COALESCE(sc.emblem_color, 'unknown') AS color_group,
            AVG(COALESCE(sp.base_points, 0.0)) AS points_x1
        FROM target_maps tm
        JOIN role_players rp
          ON rp.team_name = tm.team_name
        JOIN fantasy_scoring_stat_catalog sc
          ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp
          ON sp.match_id = tm.match_id
         AND sp.account_id = rp.account_id
         AND sp.team_name = rp.team_name
         AND sp.stat_name = sc.stat_name
        GROUP BY tm.match_id, tm.team_name, sc.stat_name, sc.emblem_color
    ),
    ranked AS (
        SELECT
            rsm.team_name,
            rsm.role_slot,
            rn.player_names,
            rn.role_group,
            rsm.stat_name,
            rsm.color_group,
            rsm.points_x1,
            ROW_NUMBER() OVER (
                PARTITION BY rsm.team_name, rsm.role_slot, rsm.stat_name
                ORDER BY rsm.points_x1
            ) AS rn,
            COUNT(*) OVER (
                PARTITION BY rsm.team_name, rsm.role_slot, rsm.stat_name
            ) AS cnt
        FROM role_stat_maps rsm
        JOIN role_names rn
          ON rn.team_name = rsm.team_name
    )
    SELECT
        team_name,
        role_slot,
        player_names,
        role_group,
        stat_name,
        color_group,
        COUNT(*) AS maps_played,
        ROUND(AVG(points_x1), 2) AS avg_fantasy_points_x1,
        ROUND(MAX(points_x1), 2) AS max_fantasy_points_x1,
        ROUND(
            AVG(
                CASE
                    WHEN rn IN (
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                    )
                    THEN points_x1
                END
            ),
            2
        ) AS p75_fantasy_points_x1
    FROM ranked
    GROUP BY
        team_name,
        role_slot,
        player_names,
        role_group,
        stat_name,
        color_group
    HAVING COUNT(*) >= {int(min_maps)}
    ORDER BY stat_name, p75_fantasy_points_x1 DESC, max_fantasy_points_x1 DESC, team_name
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


def show_role_slot_stat_rankings(role_key, top_n=10, min_maps=1, ti_qualified_only=TI_QUALIFIED_ONLY):
    cfg = ROLE_RANKING_CONFIG[role_key]
    df = get_role_slot_stat_ranking(
        cfg["positions"],
        role_key=role_key,
        min_maps=min_maps,
        ti_qualified_only=ti_qualified_only,
    )
    print()
    print("=" * 24, cfg["label"], "=" * 24)
    print(f"Rows: {len(df)}")
    print(f"TI-qualified only: {ti_qualified_only}")
    for stat_name in df["stat_name"].drop_duplicates().tolist():
        chunk = (
            df[df["stat_name"] == stat_name]
            .sort_values(["p75_fantasy_points_x1", "max_fantasy_points_x1", "avg_fantasy_points_x1"], ascending=False)
            .head(top_n)
            .reset_index(drop=True)
        )
        print()
        print(f"STAT: {stat_name}")
        display(chunk)
    return df


role_slot_stat_rankings = {
    role_key: get_role_slot_stat_ranking(
        cfg["positions"],
        role_key=role_key,
        ti_qualified_only=TI_QUALIFIED_ONLY,
    )
    for role_key, cfg in ROLE_RANKING_CONFIG.items()
}
player_stat_rankings = role_slot_stat_rankings

for role_key in ROLE_RANKING_CONFIG:
    show_role_slot_stat_rankings(role_key, top_n=10, ti_qualified_only=TI_QUALIFIED_ONLY)



In [ ]:
# ## 3.3 Ранжирование командных сочетаний под мой текущий баннерный профиль и коэффициенты

import sqlite3
import pandas as pd

MY_ROLE_BANNER_STATS = {
    "core_pair": MY_CUSTOM_BANNER["core"],
    "mid_single": MY_CUSTOM_BANNER["mid"],
    "support_pair": MY_CUSTOM_BANNER["support"],
}


def get_weighted_role_slot_ranking(positions, role_key, stat_weights, min_maps=1, ti_qualified_only=TI_QUALIFIED_ONLY):
    con = sqlite3.connect(DB_PATH)
    pos_list, expected_positions, team_filter_sql = _build_role_slot_context(
        positions,
        ti_qualified_only=ti_qualified_only,
    )
    values_sql = ",\n        ".join(
        f"('{stat_name}', {float(weight)})" for stat_name, weight in stat_weights
    )
    sql = f"""
    WITH selected_stats(stat_name, weight) AS (
        VALUES
        {values_sql}
    ),
    role_players AS (
        SELECT
            pir.account_id,
            pir.team_name,
            pir.official_name,
            pir.official_position,
            pir.role_group
        FROM player_identity_registry pir
        WHERE pir.official_position IN ({pos_list})
        {team_filter_sql}
    ),
    role_names AS (
        SELECT
            rp.team_name,
            GROUP_CONCAT(rp.official_name, ', ') AS player_names,
            MIN(rp.role_group) AS role_group
        FROM (
            SELECT team_name, official_name, official_position, role_group
            FROM role_players
            ORDER BY team_name, official_position
        ) rp
        GROUP BY rp.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    target_maps AS (
        SELECT
            f.match_id,
            f.team_name
        FROM player_game_fantasy_summary f
        JOIN role_players rp
          ON rp.account_id = f.account_id
         AND rp.team_name = f.team_name
        GROUP BY f.match_id, f.team_name
        HAVING COUNT(DISTINCT rp.official_position) = {expected_positions}
    ),
    player_map_scores AS (
        SELECT
            tm.match_id,
            tm.team_name,
            rp.account_id,
            SUM(COALESCE(sp.base_points, 0.0) * ss.weight) AS weighted_points
        FROM target_maps tm
        JOIN role_players rp
          ON rp.team_name = tm.team_name
        JOIN selected_stats ss
          ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp
          ON sp.match_id = tm.match_id
         AND sp.account_id = rp.account_id
         AND sp.team_name = rp.team_name
         AND sp.stat_name = ss.stat_name
        GROUP BY tm.match_id, tm.team_name, rp.account_id
    ),
    role_map_scores AS (
        SELECT
            pms.match_id,
            pms.team_name,
            '{role_key}' AS role_slot,
            AVG(pms.weighted_points) AS weighted_points
        FROM player_map_scores pms
        GROUP BY pms.match_id, pms.team_name
    ),
    ranked AS (
        SELECT
            rms.team_name,
            rms.role_slot,
            rn.player_names,
            rn.role_group,
            rms.weighted_points,
            ROW_NUMBER() OVER (
                PARTITION BY rms.team_name, rms.role_slot
                ORDER BY rms.weighted_points
            ) AS rn,
            COUNT(*) OVER (
                PARTITION BY rms.team_name, rms.role_slot
            ) AS cnt
        FROM role_map_scores rms
        JOIN role_names rn
          ON rn.team_name = rms.team_name
    )
    SELECT
        team_name,
        role_slot,
        player_names,
        role_group,
        COUNT(*) AS maps_played,
        ROUND(AVG(weighted_points), 2) AS avg_weighted_points,
        ROUND(MAX(weighted_points), 2) AS max_weighted_points,
        ROUND(
            AVG(
                CASE
                    WHEN rn IN (
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                    )
                    THEN weighted_points
                END
            ),
            2
        ) AS p75_weighted_points
    FROM ranked
    GROUP BY team_name, role_slot, player_names, role_group
    HAVING COUNT(*) >= {int(min_maps)}
    ORDER BY p75_weighted_points DESC, max_weighted_points DESC, avg_weighted_points DESC, team_name
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


my_banner_role_rankings = {}
my_banner_player_rankings = my_banner_role_rankings
for role_key, stat_weights in MY_ROLE_BANNER_STATS.items():
    cfg = ROLE_RANKING_CONFIG[role_key]
    print()
    print("=" * 24, f"MY BANNER - {cfg['label']}", "=" * 24)
    print("Selected stats:", stat_weights)
    print(f"TI-qualified only: {TI_QUALIFIED_ONLY}")
    ranking_df = get_weighted_role_slot_ranking(
        cfg["positions"],
        role_key=role_key,
        stat_weights=stat_weights,
        ti_qualified_only=TI_QUALIFIED_ONLY,
    )
    my_banner_role_rankings[role_key] = ranking_df
    display(ranking_df.head(15))



In [ ]:
# ## 3.4 Оптимальные статы по ролям и ранжирование сочетаний игроков для них

import sqlite3
import pandas as pd

# Assumption for optimal slot templates:
# - core_pair    -> 2 red + 1 green
# - mid_single   -> 1 red + 1 blue + 1 green
# - support_pair -> 2 blue + 1 green
OPTIMAL_COLOR_TEMPLATES = {
    "core_pair": {"red": 2, "green": 1},
    "mid_single": {"red": 1, "blue": 1, "green": 1},
    "support_pair": {"blue": 2, "green": 1},
}


def get_role_stat_summary(positions, ti_qualified_only=TI_QUALIFIED_ONLY):
    con = sqlite3.connect(DB_PATH)
    pos_list = ", ".join(map(str, positions))
    count_check = f"HAVING COUNT(DISTINCT pir.official_position) = {len(positions)}"
    team_filter_sql = """
        AND EXISTS (
            SELECT 1
            FROM analytics_ti2026_teams ti
            WHERE ti.team_name = pir.team_name
        )
    """ if ti_qualified_only else ""
    sql = f"""
    WITH target_maps AS (
        SELECT
            f.match_id,
            f.team_name
        FROM player_game_fantasy_summary f
        JOIN player_identity_registry pir
          ON pir.account_id = f.account_id
         AND pir.team_name = f.team_name
        WHERE pir.official_position IN ({pos_list})
        {team_filter_sql}
        GROUP BY f.match_id, f.team_name
        {count_check}
    ),
    stat_points AS (
        SELECT
            c.match_id,
            c.team_name,
            sc.stat_name,
            COALESCE(sc.emblem_color, 'unknown') AS color_group,
            AVG(COALESCE(sp.base_points, 0.0)) AS avg_points_x1
        FROM target_maps c
        JOIN player_game_fantasy_summary f
          ON f.match_id = c.match_id
         AND f.team_name = c.team_name
        JOIN player_identity_registry pir
          ON pir.account_id = f.account_id
         AND pir.team_name = f.team_name
         AND pir.official_position IN ({pos_list})
        JOIN fantasy_scoring_stat_catalog sc
          ON 1 = 1
        LEFT JOIN fantasy_player_map_stat_points sp
          ON sp.match_id = f.match_id
         AND sp.account_id = f.account_id
         AND sp.team_name = f.team_name
         AND sp.stat_name = sc.stat_name
        GROUP BY c.match_id, c.team_name, sc.stat_name, sc.emblem_color
    ),
    ranked AS (
        SELECT
            stat_name,
            color_group,
            avg_points_x1,
            ROW_NUMBER() OVER (PARTITION BY stat_name ORDER BY avg_points_x1) AS rn,
            COUNT(*) OVER (PARTITION BY stat_name) AS cnt
        FROM stat_points
    )
    SELECT
        stat_name,
        color_group,
        COUNT(*) AS role_maps,
        ROUND(AVG(avg_points_x1), 2) AS avg_fantasy_points_x1,
        ROUND(MAX(avg_points_x1), 2) AS max_fantasy_points_x1,
        ROUND(
            AVG(
                CASE
                    WHEN rn IN (
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 1,
                        CAST(((cnt - 1) * 0.75) AS INTEGER) + 2
                    )
                    THEN avg_points_x1
                END
            ),
            2
        ) AS p75_fantasy_points_x1
    FROM ranked
    GROUP BY stat_name, color_group
    ORDER BY p75_fantasy_points_x1 DESC, max_fantasy_points_x1 DESC
    """
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


def select_optimal_stats(role_key, ti_qualified_only=TI_QUALIFIED_ONLY):
    cfg = ROLE_RANKING_CONFIG[role_key]
    summary = get_role_stat_summary(
        cfg["positions"],
        ti_qualified_only=ti_qualified_only,
    )
    template = OPTIMAL_COLOR_TEMPLATES[role_key]
    selected = []
    for color_group, take_n in template.items():
        chunk = (
            summary[(summary["color_group"] == color_group) & (summary["p75_fantasy_points_x1"] > 0)]
            .sort_values(["p75_fantasy_points_x1", "max_fantasy_points_x1", "avg_fantasy_points_x1"], ascending=False)
            .head(take_n)
            .copy()
        )
        chunk["selected_weight"] = 1.0
        selected.append(chunk)
    selected_df = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame()
    return summary, selected_df


optimal_role_stat_summaries = {}
optimal_role_selected_stats = {}
optimal_role_combo_rankings = {}
optimal_role_player_rankings = optimal_role_combo_rankings

for role_key in ROLE_RANKING_CONFIG:
    cfg = ROLE_RANKING_CONFIG[role_key]
    summary_df, selected_df = select_optimal_stats(
        role_key,
        ti_qualified_only=TI_QUALIFIED_ONLY,
    )
    optimal_role_stat_summaries[role_key] = summary_df
    optimal_role_selected_stats[role_key] = selected_df

    print()
    print("=" * 24, f"OPTIMAL STATS - {cfg['label']}", "=" * 24)
    print("Color template:", OPTIMAL_COLOR_TEMPLATES[role_key])
    print(f"TI-qualified only: {TI_QUALIFIED_ONLY}")
    display(selected_df.reset_index(drop=True))

    stat_weights = [(row.stat_name, 1.0) for row in selected_df.itertuples()]
    ranking_df = (
        get_weighted_role_slot_ranking(
            cfg["positions"],
            role_key=role_key,
            stat_weights=stat_weights,
            ti_qualified_only=TI_QUALIFIED_ONLY,
        )
        if stat_weights else pd.DataFrame()
    )
    optimal_role_combo_rankings[role_key] = ranking_df

    print()
    print(f"COMBINATIONS FOR OPTIMAL STATS - {cfg['label']}")
    display(ranking_df.head(15))

